In [1]:
!pip install --upgrade langchain langchain-community langgraph openai langchain_openai wikipedia -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi 0.115.6 requires starlette<0.42.0,>=0.40.0, but you have starlette 0.52.1 which is incompatible.
langchain-groq 0.2.3 requires langchain-core<0.4.0,>=0.3.29, but you have langchain-core 1.2.23 which is incompatible.
instructor 1.12.0 requires jiter<0.11,>=0.6.1, but you have jiter 0.13.0 which is incompatible.
instructor 1.12.0 requires openai<2.0.0,>=1.70.0, but you have openai 2.30.0 which is incompatible.
langchain-ollama 0.3.3 requires langchain-core<1.0.0,>=0.3.60, but you have langchain-core 1.2.23 which is incompatible.
crewai 1.8.1 requires openai~=1.83.0, but you have openai 2.30.0 which is incompatible.
crewai 1.8.1 requires tokenizers~=0.20.3, but you have tokenizers 0.22.2 which is incompatible.
langgraph-supervisor 0.0.29 requires langgraph<0.7.0,>=0.6.0, but you have langgraph 1.1.3 which is 

In [2]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=300)
# top_k_results: The number of search results to return from the Wikipedia API. Default is 3.
# doc_content_chars_max: The maximum number of characters to return from the content of each Wikipedia page. Default is 5000.
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper) # Create a WikipediaQueryRun tool using the WikipediaAPIWrapper instance. This tool can be used to query Wikipedia for information.

In [3]:
wiki_tool.run({"query": "AI agents"})

'Page: AI agent\nSummary: In the context of generative artificial intelligence, AI agents (also referred to as compound AI systems or agentic AI) are a class of intelligent agents distinguished by their ability to operate autonomously in complex environments. Agentic AI tools prioritize decision-makin'

In [4]:
from langchain_openai import ChatOpenAI
import os
llm = ChatOpenAI(temperature=0, api_key=os.environ["OPENAI_API_KEY"], model="gpt-4o-mini")


/Users/ingledarshan/.local/share/virtualenvs/AH-KbrtVpZi/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
tools = [wiki_tool]

# Tool binding
llm_with_tools = llm.bind_tools(tools)

#Tool calling
result = llm_with_tools.invoke("Hello world!")
result
result.content

'Hello! How can I assist you today?'

In [6]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(llm, tools)

/var/folders/5g/9xpg7d6d4114s98tv10y9rgm0000gn/T/ipykernel_78031/2344901266.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


In [7]:
from langchain_core.messages import HumanMessage

#First up, let's see how it responds when there's no need to call a tool:
response = agent_executor.invoke({"messages": [HumanMessage(content="hi!")]})

response["messages"]

[HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}, id='147ba2aa-bee5-4537-a2df-28142a8df4ad'),
 AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 83, 'total_tokens': 93, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DOm3qrIVe9h9f4pA1XDwH8BBEyAPZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d3a24-e493-7143-94e8-253c3bd34f45-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 83, 'output_tokens': 10, 'total_tokens': 93, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 

In [8]:
print(response["messages"][-1].content)

Hello! How can I assist you today?


In [9]:
from langchain_core.messages import HumanMessage

#First up, let's see how it responds when there's no need to call a tool:
response = agent_executor.invoke({"messages": [HumanMessage(content="what is agentic ai")]})

response["messages"]

[HumanMessage(content='what is agentic ai', additional_kwargs={}, response_metadata={}, id='b331f8e9-0f9f-4e8a-9764-461d6d5a21e3'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 86, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DOm49v3sKz2rxdcWMJgXxsIwpLfY6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d3a25-2ec5-7261-8c6f-cd0d32679266-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'Agentic AI'}, 'id': 'call_Ohb4rJrEF41nHecjvLi65fVk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 86, 'output_tokens': 15, 'total

In [10]:
print(response["messages"][-1].content)

Agentic AI, also known as AI agents or compound AI systems, refers to a class of intelligent agents that can operate autonomously in complex environments. These systems are characterized by their ability to prioritize decision-making and take actions independently, making them capable of handling tasks without direct human intervention.


# Happy Learning